# Lesson 04 · Hypothesis testing and the false discovery rate

**Source:** *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 5 (pp. 36–44)

**Question:** We want a "Hall of Fame" of players whose true batting probability is above .300. How many can we admit while keeping wrong admissions under 5%?

**Goal:** Compute each player's posterior error probability (PEP), turn PEPs into q-values, and use them to control the false discovery rate.

**Contents**

1. Setup
2. Posterior error probabilities
3. False discovery rate of a chosen group
4. q-values
5. PEP vs q-value
6. Exercises: the frequentist route

**Tags:** `hypothesis-testing` `posterior-error-probability` `false-discovery-rate` `q-values` `multiple-testing`

**Before you start:** Lesson 03.

*How this notebook works:* each **Your turn** prompt is followed by an empty cell for your code. **Check** boxes give values to compare against; they come from the book, and your numbers can differ slightly because the Lahman data now runs through more recent seasons. **Reflect** prompts are for short written answers. Every prompt is numbered *lesson.section.question* (for example **2.3.1**), so you can refer to it; empty code cells and answer cells carry the same number.

In [ ]:
# --- Setup: run this cell first ------------------------------------------
# Windows + conda: put the environment's DLL folders on PATH, so plotting
# can't crash the kernel when Jupyter wasn't started from the activated env.
import os, sys
for sub in ["Library\\mingw-w64\\bin", "Library\\usr\\bin", "Library\\bin", "Scripts", "bin"]:
    p = os.path.join(sys.prefix, sub)
    if os.path.isdir(p) and p not in os.environ["PATH"]:
        os.environ["PATH"] = p + os.pathsep + os.environ["PATH"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import stats, optimize, special

import eb_data   # helpers in this folder: loading the Lahman data, saving results

rng = np.random.default_rng(2017)
pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (7, 4)
print(f"numpy {np.__version__} | pandas {pd.__version__} | scipy {scipy.__version__}")

career = eb_data.career(pitcher_rule=None)
alpha0, beta0 = eb_data.load_prior()
print(f"{len(career):,} players | prior alpha0 = {alpha0:.1f}, beta0 = {beta0:.1f}")

## 1. Setup

**4.1.1 · Your turn.** Add `eb_estimate`, `alpha1` and `beta1` to `career`, as in Lesson 03.

In [ ]:
# 4.1.1 · Your code here

## 2. Posterior error probabilities

For our Hall of Fame, the error is admitting a player whose true average is *below* .300. The **posterior error probability** is the posterior probability of exactly that:

$$\text{PEP}_i = \Pr(p_i < 0.3 \mid H_i, AB_i) = F_{\text{Beta}(\alpha_{1,i},\, \beta_{1,i})}(0.3).$$

**4.2.1 · Your turn.** Find Hank Aaron's posterior, plot it, and shade the area below .300. Compute his PEP with `stats.beta.cdf`.

In [ ]:
# 4.2.1 · Your code here

> **Check:** Hank Aaron's PEP ≈ 0.17–0.19, depending on your prior.

**4.2.2 · Your turn.** Compute `PEP` for every player. Plot a histogram of PEPs, and a scatter of `eb_estimate` against `PEP` colored by log AB.

In [ ]:
# 4.2.2 · Your code here

**4.2.3 · Reflect.** Why does the PEP cross 0.5 right where the shrunken estimate crosses .300? And why can two players with the same shrunken estimate have very different PEPs?

*4.2.3 · Your answer:*

## 3. False discovery rate of a chosen group

If we admit a group of players, the expected number of mistakes is the sum of their PEPs, so the expected *fraction* of mistakes is their mean PEP.

**4.3.1 · Your turn.** Sort by PEP. For the top 50, 100 and 200 players, compute the expected number of false discoveries and the mean PEP. Look at the players ranked 90–100.

In [ ]:
# 4.3.1 · Your code here

> **Check:** top 100 → mean PEP ≈ 0.05; top 50 → ≈ 0.002; top 200 → ≈ 0.27.

## 4. q-values

Doing that for every cutoff at once gives the **q-value**: the cumulative mean of the sorted PEPs. Admitting everyone with `qvalue < 0.05` keeps the expected false discovery rate below 5%.

**4.4.1 · Your turn.** Add a `qvalue` column (sort by PEP, then `.expanding().mean()` or `np.cumsum(...) / np.arange(1, n + 1)`). How many players have q < 0.05? And q < 0.01?

In [ ]:
# 4.4.1 · Your code here

> **Check:** the book got 97 players at 5% and 64 at 1%.

**4.4.2 · Your turn.** Plot the number of players admitted against the q-value threshold, for thresholds from 0 to 0.3.

In [ ]:
# 4.4.2 · Your code here

## 5. PEP vs q-value

**4.5.1 · Reflect.** Hank Aaron's PEP is about 17%, yet he gets in under a 5% FDR. Explain how, in your own words. The book compares it to keeping a group's *average height* above a target.

*4.5.1 · Your answer:*

## 6. Exercises: the frequentist route

**4.6.1 · Your turn.** For every player, compute a one-sided binomial p-value for H₀: p ≤ 0.300 (`stats.binom.sf(H - 1, AB, 0.3)`). Apply Benjamini–Hochberg with `stats.false_discovery_control` and admit those with adjusted p < 0.05. How does this list compare with the q-value list? Which players differ, and why?

In [ ]:
# 4.6.1 · Your code here

**4.6.2 · Your turn.** **Card connection.** Use your simulated Pokémon card sets from 2.8.2 or 3.5.2. Each has a *pull rate*: the chance that one booster pack contains a rare card, a *hit*. Collectors often quote a rule of thumb like "one hit every 20 packs". Flag the sets whose pull rate is above 1 in 20 while keeping the FDR at 10%.

In [ ]:
# 4.6.2 · Setup: load the card sets (with alpha1, beta1) saved after 3.5.2, and the card prior
sets = eb_data.load_saved_table("card_sets_posterior")
p = eb_data.load_result("card_prior")
alpha_pull, beta_pull = p["alpha0"], p["beta0"]
print(f"Card prior: Beta({alpha_pull:.2f}, {beta_pull:.2f})")
sets.head()

In [ ]:
# 4.6.2 · Your code here

In [ ]:
# 4.6.2 · Save the sets with your FDR results for later lessons
eb_data.save_table("card_sets_fdr", sets)

## References

- *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 5.
- Storey (2002), "A direct approach to false discovery rates", *JRSS B* 64(3).
- Käll, Storey, MacCoss & Noble (2008), "Posterior error probabilities and false discovery rates: two sides of the same coin", *J. Proteome Research* 7(1).
- SciPy `false_discovery_control`: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.false_discovery_control.html